# Step 03 – Predictor screening on the training partition only

Paper 2: *Benchmarking Ten Machine-Learning Algorithms for Mediterranean Forest Mapping with Sentinel-2* (Remote Sensing, 2026).

- **In the paper:** Section 2.3 · Table 1, Table S2
- **Needs:** training-sample table in `DATA_DIR`
- **Writes:** `RESULTS_DIR/correlation_training_only/`

Run the configuration cell below first.

In [ ]:
# ---------------------------------------------------------------- paths
# The data are not included in this repository. Set the two folders below to
# where the files are on your computer (or on your Google Drive in Colab).
#   DATA_DIR    : input data (training samples, Sentinel-2 stack, map tiles, rasters)
#   RESULTS_DIR : outputs written by the notebooks. Later steps read the outputs
#                 of earlier ones (e.g. RESULTS_DIR/train_test_gap/ from step 04).
import os
DATA_DIR    = 'data/'
RESULTS_DIR = 'results/'

# Google Colab: mount Drive and point the folders to it, for example
# from google.colab import drive; drive.mount('/content/drive')
# DATA_DIR    = '/content/drive/MyDrive/<your_folder>/'
# RESULTS_DIR = '/content/drive/MyDrive/<your_folder>/results/'

os.makedirs(RESULTS_DIR, exist_ok=True)

# Files and folders used by this notebook:
#   DATA_DIR / S2_Features_For_Correlation_v3.csv
#   RESULTS_DIR / correlation_training_only/

In [ ]:
# ============================================================
# Paper 2 (remotesensing-4538333) — v5 (for the corrected GEE export; joint correlation + VIF forward selection)
# Predictor screening on the training partition
# Training-only screening with an explicit retention rule, then VIF
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import f_classif
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

CSV_PATH   = DATA_DIR + 'S2_Features_For_Correlation_v3.csv'   # corrected GEE export
OUT_DIR    = RESULTS_DIR + 'correlation_training_only/'
CLASS_COL  = 'class'
SEED       = 42          # same random_state as the 70/30 split used for the models
TEST_SIZE  = 0.30
R_THRESH   = 0.80
VIF_THRESH = 10
EXTRA_DROP = ['latitude', 'longitude']   # coordinates are not model predictors
FIX_INDICES = False      # indices are now correct at source (GEE); keep False so training data = map data
TEX_SIDES  = (7, 11)     # GLCM window sides used in the GEE export (SIDES there); smaller one gets priority
TIERED     = True        # True: type priority, then ANOVA F within type; False: ANOVA F only
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_csv(CSV_PATH)
df = df.drop(columns=[c for c in ['system:index', '.geo'] + EXTRA_DROP if c in df.columns])
y = df[CLASS_COL]
X = df.drop(columns=[CLASS_COL]).select_dtypes(include=[np.number])
print(f'{X.shape[1]} candidate predictors, {len(X)} samples')

# ---- helpers --------------------------------------------------
def correlated_pairs(corr, thr):
    upper = corr.abs().where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
    s = upper.stack().dropna()
    s = s[s >= thr].sort_values(ascending=False)
    out = s.reset_index()
    out.columns = ['feature_1', 'feature_2', 'abs_r']
    out['r'] = [round(corr.loc[a, b], 3) for a, b in zip(out.feature_1, out.feature_2)]
    out['abs_r'] = out['abs_r'].round(3)
    return out

def vif_table(Xs):
    Z = (Xs - Xs.mean()) / Xs.std(ddof=0)
    Zc = add_constant(Z, has_constant='add')
    with warnings.catch_warnings(), np.errstate(divide='ignore'):
        warnings.simplefilter('ignore')
        v = [variance_inflation_factor(Zc.values, i) for i in range(1, Zc.shape[1])]
    return (pd.DataFrame({'feature': Xs.columns, 'VIF': np.round(v, 2)})
              .sort_values('VIF', ascending=False).reset_index(drop=True))

# ---- STEP 1: formula checks (EVI, SAVI, SATVI, IBI, UI) -------
print('\n=== STEP 1: formula checks ===')
print(f"B8 range: {X['B8'].min():.4f} – {X['B8'].max():.4f}  (max > 1.5 means 0–10000 scaling)")
print(X[['NDVI', 'EVI', 'SAVI', 'IBI', 'SATVI', 'UI']].describe().T.round(4).to_string())
scale = 10000.0 if X['B8'].max() > 1.5 else 1.0
b2, b4, b8 = X['B2'] / scale, X['B4'] / scale, X['B8'] / scale
evi_ref = 2.5 * (b8 - b4) / (b8 + 6 * b4 - 7.5 * b2 + 1)
print(f"Pearson  EVI(file) vs NDVI          : {X['EVI'].corr(X['NDVI']):.3f}")
print(f"Spearman EVI(file) vs NDVI          : {X['EVI'].corr(X['NDVI'], method='spearman'):.3f}")
print(f"Pearson  EVI(recomputed) vs NDVI    : {evi_ref.corr(X['NDVI']):.3f}")
print(f"Pearson  EVI(file) vs EVI(recomputed): {X['EVI'].corr(evi_ref):.3f}")
print(f"Samples with B8 = 0: {(X['B8'] == 0).sum()} | classes: {y[X['B8'] == 0].value_counts().to_dict()}")

# Consistency check: GEE indices vs reference formulas (all should print 1.000)
_s = 10000.0 if X['B8'].max() > 1.5 else 1.0
_B, _G, _R, _N = X['B2'] / _s, X['B3'] / _s, X['B4'] / _s, X['B8'] / _s
_S1, _S2 = X['B11'] / _s, X['B12'] / _s
_ref = {
    'EVI':   2.5 * (_N - _R) / (_N + 6 * _R - 7.5 * _B + 1),
    'SAVI':  1.5 * (_N - _R) / (_N + _R + 0.5),
    'SATVI': 1.5 * (_S1 - _R) / (_S1 + _R + 0.5) - _S2 / 2,
    'IBI':   ((2 * _S1 / (_S1 + _N)) - (_N / (_N + _R) + _G / (_G + _S1)))
             / ((2 * _S1 / (_S1 + _N)) + (_N / (_N + _R) + _G / (_G + _S1))),
    'UI':    (_S2 - _N) / (_S2 + _N),
    'RENDVI': (X['B6'] - X['B5']) / (X['B6'] + X['B5']),
}
print('\nGEE vs reference formula (Pearson r):')
for k, v in _ref.items():
    print(f'  {k:7s}: {X[k].corr(v):.3f}')
print(f"  RENDVI vs NDRE (should now be < 1): {X['RENDVI'].corr(X['NDRE']):.3f}")

if FIX_INDICES:
    # Normalised differences (NDVI, NDWI, NDMI, NDBI, BSI, ...) are scale-invariant.
    # These four contain additive constants or raw-band terms, so they need 0–1 reflectance.
    s = scale
    B, G, R, N = X['B2'] / s, X['B3'] / s, X['B4'] / s, X['B8'] / s
    S1, S2 = X['B11'] / s, X['B12'] / s
    X['EVI']   = 2.5 * (N - R) / (N + 6 * R - 7.5 * B + 1)
    X['SAVI']  = 1.5 * (N - R) / (N + R + 0.5)
    X['SATVI'] = 1.5 * (S1 - R) / (S1 + R + 0.5) - S2 / 2
    ibi_a = 2 * S1 / (S1 + N)                    # Xu (2008), band form, bounded [-1, 1]
    ibi_b = N / (N + R) + G / (G + S1)
    X['IBI']   = (ibi_a - ibi_b) / (ibi_a + ibi_b)

    X = X.replace([np.inf, -np.inf], np.nan)
    n_bad = int(X.isna().any(axis=1).sum())
    print(f'\n>> EVI, SAVI, SATVI, IBI recomputed on 0–1 reflectance. Non-finite rows: {n_bad}')
    assert n_bad == 0, 'Some recomputed values are NaN/inf: inspect those rows before continuing.'
    print(X[['NDVI', 'EVI', 'SAVI', 'IBI', 'SATVI']].describe().T.round(4).to_string())
    print(f"Pearson EVI(new) vs NDVI : {X['EVI'].corr(X['NDVI']):.3f}")
    print(f"Pearson SAVI(new) vs NDVI: {X['SAVI'].corr(X['NDVI']):.3f}")

# ---- split (same as modelling) --------------------------------
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED)
corr = X_tr.corr(method='pearson')
print(f'\nTraining partition: {X_tr.shape}')

# ---- STEP 3: explicit retention rule on the training partition -
# 3a. Exact redundancies (|r| ~ 1): removed first, retained member stated.
s0, s1 = TEX_SIDES
REDUNDANT = {   # removed: (duplicate of, reason)
    'NDBI':  ('NDMI', 'same B8/B11 normalised difference, opposite sign'),
    'GNDVI': ('NDWI', 'same B3/B8 normalised difference, opposite sign'),
    f'NDVI_inertia_{s0}x{s0}': (f'NDVI_contrast_{s0}x{s0}', 'identical GLCM formula'),
    f'NDVI_inertia_{s1}x{s1}': (f'NDVI_contrast_{s1}x{s1}', 'identical GLCM formula'),
}

# 3b. Priority by predictor type, then class separability (ANOVA F) within type.
def tier(f):
    if f in ('elevation', 'slope'):
        return 0                                   # terrain
    if f.startswith('NDVI_') and f.endswith(f'_{s0}x{s0}'):
        return 3                                   # texture, smaller window
    if f.startswith('NDVI_') and f.endswith(f'_{s1}x{s1}'):
        return 4                                   # texture, larger window
    if f.startswith('B') and f[1:].rstrip('A').isdigit():
        return 2                                   # spectral band
    return 1                                       # spectral index

TIER_NAME = {0: 'terrain', 1: 'spectral index', 2: 'spectral band',
             3: f'texture {s0}x{s0}', 4: f'texture {s1}x{s1}'}

cand = [f for f in X_tr.columns if f not in REDUNDANT]
F = pd.Series(f_classif(X_tr[cand], y_tr)[0], index=cand).fillna(0)
order = sorted(cand, key=lambda f: ((tier(f) if TIERED else 0), -F[f]))

log = []
for f, (k, why) in REDUNDANT.items():
    if f in X_tr.columns:
        log.append({'feature': f, 'type': TIER_NAME[tier(f)], 'decision': 'removed',
                    'stage': 'exact redundancy', 'duplicate_of': k,
                    'abs_r': round(abs(corr.loc[f, k]), 3), 'note': why})

kept = []
for f in order:
    # (i) pairwise rule: |r| < R_THRESH with every predictor already kept
    clash = [k for k in kept if abs(corr.loc[f, k]) >= R_THRESH]
    if clash:
        k = max(clash, key=lambda c: abs(corr.loc[f, c]))
        log.append({'feature': f, 'type': TIER_NAME[tier(f)], 'F': round(F[f], 1),
                    'decision': 'removed', 'stage': 'correlation', 'retained_instead': k,
                    'abs_r': round(abs(corr.loc[f, k]), 3), 'F_retained': round(F[k], 1)})
        continue
    # (ii) multivariate rule: adding f must keep every VIF < VIF_THRESH
    if len(kept) >= 2:
        v = vif_table(X_tr[kept + [f]])
        if v.VIF.iloc[0] >= VIF_THRESH:
            log.append({'feature': f, 'type': TIER_NAME[tier(f)], 'F': round(F[f], 1),
                        'decision': 'removed', 'stage': 'VIF',
                        'VIF': float(v.set_index('feature').VIF[f]),
                        'max_VIF_if_added': float(v.VIF.iloc[0]),
                        'max_VIF_feature': v.feature.iloc[0]})
            continue
    kept.append(f)

# Because a predictor is only added when both rules hold, no predictor is removed
# after it has already excluded its correlated partners (no orphaned groups).

final_vif = vif_table(X_tr[kept])
for f in kept:
    log.append({'feature': f, 'type': TIER_NAME[tier(f)], 'F': round(F[f], 1),
                'decision': 'retained', 'stage': '-',
                'VIF': final_vif.set_index('feature').VIF[f]})

log = pd.DataFrame(log)
max_r = corr.loc[kept, kept].abs().where(~np.eye(len(kept), dtype=bool)).max().max()

print('\n=== STEP 3: new predictor set (training partition) ===')
print(f'Retained: {len(kept)} | max |r| = {max_r:.3f} | max VIF = {final_vif.VIF.max()}')
print(log[log.decision == 'retained'].groupby('type').feature.apply(list).to_string())
print('\nRemoved at each stage:', log[log.decision == 'removed'].stage.value_counts().to_dict())
print('\nFinal VIF:')
print(final_vif.to_string(index=False))
print('\nCopy into the modelling notebook:')
print('PREDICTORS =', kept)

cols = [c for c in ['feature', 'F', 'decision', 'stage', 'duplicate_of', 'retained_instead', 'abs_r',
                    'max_VIF_if_added', 'max_VIF_feature'] if c in log.columns]
print('\nDecisions for spectral indices and bands (in ranking order):')
ib = log[log.type.isin(['spectral index', 'spectral band'])].copy()
ib['rank'] = ib.feature.map({f: i for i, f in enumerate(order)})
print(ib.sort_values('rank')[cols].to_string(index=False))

log.to_csv(OUT_DIR + 'v5_selection_log_training.csv', index=False)
final_vif.to_csv(OUT_DIR + 'v5_final_vif_training.csv', index=False)
print(f'\nFiles saved to: {OUT_DIR}')